# Lamalo Free Video-Ready Wardrobe Generation

Generates the Lamalo clothing collection with FLUX.1-schnell, TRELLIS (preferred), TripoSR fallback, Qwen3-VL visual QA and Blender. No paid generation API, database credential, Render credential, storage credential or DNS change is required. Enable a Kaggle GPU and Internet before running.


In [ ]:
START_ORDINAL = 1
COUNT = 149          # all current Lamalo base designs
PARITY = 'all'       # all, odd, even
FORCE = False        # keep False so approved work is reused


In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/kaggle/working/virellestudios')
if repo.exists():
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','https://github.com/leego972/virellestudios.git',str(repo)], check=True)
os.chdir(repo)
subprocess.run(['bash','scripts/lamalo360/free/bootstrap_kaggle.sh'], check=True)
subprocess.run(['corepack','enable'], check=True)
subprocess.run(['corepack','prepare','pnpm@10.4.1','--activate'], check=True)
subprocess.run(['pnpm','install','--frozen-lockfile'], check=True)


In [ ]:
# HF_TOKEN is optional and free. It is needed only if a model repository asks you to accept its terms.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token = None
if token:
    os.environ['HF_TOKEN'] = token
os.environ['LAMALO360_WORK_ROOT'] = '/kaggle/working/lamalo360-work'
os.environ['TRIPOSR_HOME'] = '/kaggle/working/TripoSR'
os.environ['TRELLIS_HOME'] = '/kaggle/working/TRELLIS'
os.environ['BLENDER_BIN'] = subprocess.check_output(['which','blender'], text=True).strip()
os.environ['LAMALO_FREE_3D_ENGINE'] = 'auto'
os.environ['LAMALO360_CYCLES_SAMPLES'] = '256'


In [ ]:
subprocess.run(['node','scripts/lamalo360/catalogue.mjs'], check=True)
cmd = ['node','scripts/lamalo360/run-batch.mjs','--parity',PARITY,'--count',str(COUNT),'--start',str(START_ORDINAL),'--retry-failed','--skip-publish']
if FORCE:
    cmd.append('--force')
subprocess.run(cmd, check=True)


In [ ]:
import json, shutil, pathlib
work = pathlib.Path('/kaggle/working/lamalo360-work')
output = pathlib.Path('/kaggle/working/lamalo-video-ready-assets')
output.mkdir(parents=True, exist_ok=True)
shutil.copy2(repo / 'docs/lamalo-clothing-360-production.json', output / 'catalogue-manifest.json')
archive = shutil.make_archive(str(output / 'lamalo-collection'), 'zip', str(work))
print('Generation complete.')
print('Download this file from the Kaggle Output panel:', archive)
print('The archive contains hidden GLBs, continuity references, verification renders, textures and QA reports. The website shop uses only each SKU primary image.')


## Free-session checkpoint

A full 149-design catalogue may exceed one Kaggle session. The work directory is resumable during the active session. Before the session expires, use **Save Version → Save & Run All** so Kaggle keeps the notebook output, then continue from the next ordinal in a later free session. No paid service is required.
